<a href="https://colab.research.google.com/github/AbdlHamd/gomycode/blob/Python/y_data_profiling2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analyse des Retards de Vol de Tunisair

Ce notebook est dédié à l'exploration et à l'analyse des facteurs influençant les retards des vols de la compagnie Tunisair en utilisant **Pandas** et **ydata-profiling**.

In [1]:
!pip install -q ydata-profiling

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.8/400.8 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 85.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 682.5/682.5 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 6.2 MB/s eta 0:00:00


## Partie 1 : Exploration de données avec Pandas

Nous allons d'abord charger et effectuer une première exploration des données de Tunisair.

In [2]:
import pandas as pd
import numpy as np

# Téléchargement ou simulation des données historiques Tunisair pour l'exercice
try:
    # Si un fichier CSV est localement présent ou téléchargeable, remplacez cette URL
    url = "https://raw.githubusercontent.com/datasets/flight-delay/master/data/flights.csv"
    df = pd.read_csv(url, nrows=10000)
except Exception:
    # Simulation d'un jeu de données Tunisair si l'URL n'est pas accessible
    np.random.seed(42)
    n_rows = 1000
    data = {
        'ID_Vol': [f'TU{100 + i}' for i in range(n_rows)],
        'Date': pd.date_range(start='2023-01-01', periods=n_rows, freq='h'),
        'Aeroport_Depart': np.random.choice(['TUN', 'DJE', 'SFA', 'CDG', 'FRA'], n_rows),
        'Aeroport_Arrivee': np.random.choice(['TUN', 'CDG', 'FRA', 'LYS', 'MAR'], n_rows),
        'Retard_Depart_Min': np.random.exponential(scale=15, size=n_rows).round().astype(int),
        'Retard_Arrivee_Min': lambda x: x + np.random.normal(5, 5, n_rows).round().astype(int),
        'Statut': np.random.choice(['A l\'heure', 'Retardé', 'Annulé'], n_rows, p=[0.7, 0.25, 0.05])
    }
    # Correction du retard d'arrivée basé sur le retard de départ
    data['Retard_Arrivee_Min'] = (data['Retard_Depart_Min'] + np.random.normal(5, 5, n_rows)).round().astype(int)
    df = pd.DataFrame(data)

# Affichage des premières lignes
display(df.head())

,ID_Vol,Date,Aeroport_Depart,Aeroport_Arrivee,Retard_Depart_Min,Retard_Arrivee_Min,Statut
0,TU100,2023-01-01 00:00:00,CDG,LYS,14,19,A l'heure
1,TU101,2023-01-01 01:00:00,FRA,LYS,12,26,A l'heure
2,TU102,2023-01-01 02:00:00,SFA,LYS,4,15,A l'heure
3,TU103,2023-01-01 03:00:00,FRA,LYS,7,12,Retardé
4,TU104,2023-01-01 04:00:00,FRA,MAR,14,24,A l'heure


In [3]:
# Vue d'ensemble des types de données et valeurs manquantes
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   ID_Vol              1000 non-null   object        
 1   Date                1000 non-null   datetime64[ns]
 2   Aeroport_Depart     1000 non-null   object        
 3   Aeroport_Arrivee    1000 non-null   object        
 4   Retard_Depart_Min   1000 non-null   int64         
 5   Retard_Arrivee_Min  1000 non-null   int64         
 6   Statut              1000 non-null   object        
dtypes: datetime64[ns](1), int64(2), object(4)
memory usage: 54.8+ KB
None


In [4]:
# Recherche approfondie des valeurs manquantes
valeurs_manquantes = df.isnull().sum()
print("Nombre de valeurs manquantes par colonne :\n", valeurs_manquantes)

Nombre de valeurs manquantes par colonne :
 ID_Vol                0
Date                  0
Aeroport_Depart       0
Aeroport_Arrivee      0
Retard_Depart_Min     0
Retard_Arrivee_Min    0
Statut                0
dtype: int64


In [5]:
# Statistiques descriptives pour les colonnes numériques
display(df.describe())

,Date,Retard_Depart_Min,Retard_Arrivee_Min
count,1000,1000.00000,1000.000000
mean,2023-01-21 19:29:59.999999744,15.02900,19.914000
min,2023-01-01 00:00:00,0.00000,-5.000000
25%,2023-01-11 09:45:00,4.75000,9.000000
50%,2023-01-21 19:30:00,11.00000,16.000000
75%,2023-02-01 05:15:00,21.00000,26.000000
max,2023-02-11 15:00:00,96.00000,102.000000
std,NaN,15.03683,15.991946


## Partie 2 : Exploration des données avec ydata-Profiling

Générons maintenant un rapport complet pour identifier plus facilement les corrélations, distributions et valeurs aberrantes.

In [6]:
from ydata_profiling import ProfileReport

# Génération du profil de données
profile = ProfileReport(df, title="Rapport d'Analyse Tunisair", explorative=True)

# Pour afficher le rapport directement dans Colab
profile.to_notebook_iframe()

/tmp/ipykernel_4130/2590653560.py:1: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 7/7 [00:00<00:00, 35.40it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]